In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import math
import matplotlib.pyplot as plt
import datetime

In [2]:
tkel_bs = 11
puppi_bs = 14
nlargestpt_per_puppipart = 5
npuppipart = 5
puppicands = nlargestpt_per_puppipart * npuppipart

In [6]:
train_path = 'train_data.pt'
test_path = 'test_data.pt'

In [7]:
train_data = torch.load(train_path, weights_only=True)
test_data = torch.load(test_path, weights_only=True)

x_train, y_train = train_data['x'], train_data['y'].float()
x_test, y_test = test_data['x'], test_data['y'].float()


In [10]:
# Count the number of tkel features in data
data_features = train_data['features']
tkel_index = sum([feature.startswith('tkel') for feature in data_features])
if tkel_index != tkel_bs:
    raise(f'Required exact match of tkel features: {tkel_bs}, found {tkel_index}')
puppi_features = sum([feature.startswith('mpuppi') for feature in data_features])
if puppi_features // puppicands != puppi_bs:
    raise(f'Required exact match of puppi features: {puppi_bs}, '\
            'found {puppi_features}')

# Split into tkel and puppi
tkel_train = x_train[:, 0:tkel_bs]
puppi_train_flat = x_train[:, tkel_bs:]
if puppi_features%puppicands == 0:
    puppi_train = puppi_train_flat.reshape(-1, puppicands, puppi_bs)
else:
    raise UnboundLocalError(f"Unresolved splitting of puppi cands, found "\
            f"features {puppi_features} for {puppicands} puppi canddiates")

tkel_test = x_test[:, 0:tkel_bs]
puppi_test_flat = x_test[:, tkel_bs:]
if puppi_features%puppicands == 0:
    puppi_test = puppi_test_flat.reshape(-1, puppicands, puppi_bs)
else:
    raise UnboundLocalError(f"Unresolved splitting of puppi cands, found "\
            f"features {puppi_features} for {puppicands} puppi canddiates")



In [30]:
# Extract unnormalized pT (which is the 0-th feature of the 14 puppi features)
puppi_pt_train = puppi_train[:, :, 0].clone()
puppi_pt_test = puppi_test[:, :, 0].clone()

In [31]:
# Compute Normalization statistics on Train only
tkel_mean = tkel_train.mean(dim=0)
tkel_std = tkel_train.std(dim=0)
tkel_std[tkel_std < 1e-6] = 1.0 # Prevent division by zero

# Normalize all 25 candidates uniformly
puppi_train_reshaped = puppi_train.reshape(-1, 14)
puppi_mean = puppi_train_reshaped.mean(dim=0)
puppi_std = puppi_train_reshaped.std(dim=0)
puppi_std[puppi_std < 1e-6] = 1.0

In [40]:
# Apply normalization
tkel_train_norm = (tkel_train - tkel_mean) / tkel_std
tkel_test_norm = (tkel_test - tkel_mean) / tkel_std

puppi_train_norm = (puppi_train - puppi_mean) / puppi_std
puppi_test_norm = (puppi_test - puppi_mean) / puppi_std



In [47]:
tkel_expand = tkel_train_norm.unsqueeze(1).expand(-1, puppicands, -1)

In [48]:
tkel_expand.shape

torch.Size([197754, 25, 11])

In [49]:
combined = torch.cat([tkel_expand, puppi_train_norm], dim=2)

In [52]:
combined[0]

tensor([[ 2.5519e-01, -1.7602e+00, -1.3970e+00, -1.7806e+00, -1.4104e+00,
          6.1801e-01, -1.7584e+00, -1.3970e+00,  9.9164e-01,  3.8590e-01,
         -5.0577e-01,  5.5247e+00, -5.2216e+00, -4.1746e+00, -1.2622e-01,
          4.2718e+00,  4.1549e+00,  4.1549e+00,  5.8785e+00, -3.3901e-01,
          3.1566e+00, -2.4641e+00, -2.3206e-01,  1.7890e-03, -7.5297e-03],
        [ 2.5519e-01, -1.7602e+00, -1.3970e+00, -1.7806e+00, -1.4104e+00,
          6.1801e-01, -1.7584e+00, -1.3970e+00,  9.9164e-01,  3.8590e-01,
         -5.0577e-01, -2.1907e-01, -1.4352e-03, -2.5654e-03, -1.9546e-01,
         -1.7600e-03, -2.4068e-01, -2.4068e-01, -2.1435e-01, -8.0158e-02,
         -3.3842e-01, -3.0095e-04, -2.3743e-01,  1.5743e-04, -5.1130e-04],
        [ 2.5519e-01, -1.7602e+00, -1.3970e+00, -1.7806e+00, -1.4104e+00,
          6.1801e-01, -1.7584e+00, -1.3970e+00,  9.9164e-01,  3.8590e-01,
         -5.0577e-01, -2.1907e-01, -1.4352e-03, -2.5654e-03, -1.9546e-01,
         -1.7600e-03, -2.4068e-01, -

In [51]:
puppi_train_norm.shape

torch.Size([197754, 25, 14])